In [1]:
# 01 · STATE STAGE ACT 단일 정책 CONFIG · GitHub 중단 복구 · Drive 미사용
CFG = {
    # 저장 · ver2는 기존 DP 결과와 별도 Release에 저장
    'run_name': 'moveboxes_stage_chunk_deadline_v1',
    'profile': 'benchmark',
    'github_repository': 'SongYunu/moveBoxes',
    'project_dir': '/content/moveBoxes',
    'project_ref': 'stage-act-chunk-compare',
    'output_root': '/content/moveboxes_runs',

    # 데이터 / Colab 2026.07 · Python 3.12 · T4
    'repo_dir': '/content/berlin-marso-hackathon',
    'repo_url': 'https://github.com/marso-robotics/berlin-marso-hackathon.git',
    'repo_commit': '6048f33217f26ae39009a812f53c81171517f393',
    'data_dir': '/content/marso_data',
    'data_source': '/content/moveboxes_data_cache/marso_state_data.zip',
    'download_cache': '/content/moveboxes_data_cache',
    'packages': ['mani-skill==3.0.1', 'sapien==3.0.3', 'diffusers==0.38.0', 'hydra-core', 'omegaconf', 'gymnasium', 'tyro', 'h5py', 'kagglehub', 'tensorboard', 'matplotlib', 'transforms3d', 'imageio[ffmpeg]'],

    # 학습 · 시뮬레이터 없이 CPU 데이터 + GPU 모델
    'seed': 42,
    'num_demos': None,
    'batch_size': 64,
    'lr': 0.0001,
    'total_iters': {'easy': 12000, 'medium': 20000, 'hard': 30000},
    'amp': True,

    # 작은 State ACT · 기존 DP 체크포인트 사용 불가
    'history': 16,
    'chunk_size': 16,
    'width': 128,
    'heads': 4,
    'layers': 2,
    'latent_dim': 16,

    # 검증 / 중단 복구 / 작은 관측 위치 증강
    'save_freq': 1000,
    'warmup_steps': 500,
    'validation_batches': 8,
    'kl_weight': 0.001,
    'position_noise': 0.001,

    # 실행 · 매 스텝 재계획, 최근 XYZ 예측 평균, 집게는 최신 예측
    'temporal_decay': 0.25,
    'ensemble_window': 4,
    'ensemble_candidates': [1, 4],

    # 빠른 테스트 / 최종 평가 · 시드 분리, 기존 200스텝 유지
    'test_episodes': 8,
    'test_seed_start': 40000,
    'test_record_video': True,
    'tuning_episodes': 8,
    'tuning_seed_start': 20000,
    'benchmark_episodes': 100,
    'eval_seed_start': 30000,
    'max_episode_steps': {'easy': 200, 'medium': 200, 'hard': 200},
    'record_eval_video': True,

    # 출력
    'console_interval_seconds': 10,
    'team': 'my-team',

    # 실행 조건으로 행동 학습 · 빠른 테스트가 0이면 긴 평가 생략
    'action_training_mode': 'prior',
    'repair_iters': 2000,
    'allow_zero_success_evaluation': False,

    # 단계 판단 · 학습된 완료/복구 확신이 낮으면 현재 단계 유지
    'gate_threshold': 0.65,
    'stage_threshold': 0.6,
    'stage_loss_weight': 0.3,
    'gate_loss_weight': 0.3,

    # 복구 시연 · 수집 전용 expert, 학습/제출은 학습된 정책
    'recovery_episodes': 16,
    'recovery_max_attempts': 48,
    'recovery_seed_start': 100000,
    'collection_max_steps': {'easy': 500, 'medium': 900, 'hard': 1400},
    'noise_probability': 0.08,
    'action_noise_std': 0.12,
    'drop_probability': 0.015,

}


In [2]:
# 02 · GitHub 코드 불러오기 (데이터·결과를 위해 Drive를 마운트하지 않습니다)
import importlib, os, subprocess, sys
from pathlib import Path

PROJECT = Path(CFG['project_dir'])
URL = 'https://github.com/'+CFG['github_repository']+'.git'
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', URL, str(PROJECT)], check=True)
else:
    remote = subprocess.check_output(['git', 'remote', 'get-url', 'origin'], cwd=PROJECT, text=True).strip()
    if remote != URL:
        raise RuntimeError('기존 프로젝트 폴더가 다른 저장소입니다. project_dir를 새 경로로 바꾸세요.')
subprocess.run(['git', 'fetch', '--depth', '1', 'origin', CFG['project_ref']], cwd=PROJECT, check=True)
subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=PROJECT, check=True)
CFG['project_commit'] = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT, text=True).strip()
sys.path.insert(0, str(PROJECT))
# A fresh notebook run should not retain a previously imported project module.
for name in ('marso_experiment', 'marso_train_test', 'next_pick_sampling', 'next_pick_diagnostics',
             'marso_next_pick', 'github_store', 'github_data', 'colab_layout', 'build_modular_notebook',
             'colab_train_test_layout', 'build_train_test_notebook', 'colab_next_pick_layout',
             'build_next_pick_notebook', 'build_github_notebook', 'marso_github'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0, str(PROJECT/'ver2'))
for name in ('act_v2_model','act_v2_data','act_v2_policy','act_v2_eval','act_v2_experiment','build_act_v2_notebook'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0, str(PROJECT/'ver2'/'stages'))
for name in ('stage_schema','stage_model','stage_policy','stage_labels','stage_data','stage_teacher',
             'stage_collect','stage_eval','stage_experiment','build_stage_notebook'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
from stage_experiment import StageExperiment, source_bundle
experiment = StageExperiment(CFG, source_bundle())
print('사용 코드:', CFG['project_commit'])
print('코드 로드 완료. 03 셀로 연결하거나 다음 실행 셀에서 자동 연결합니다.')


사용 코드: af5d2cf41872b173ee3e1ef19064cb7229fb28d2
코드 로드 완료. 03 셀로 연결하거나 다음 실행 셀에서 자동 연결합니다.


In [3]:
# 03 · GitHub 인증 / 저장된 결과 복원
# 기존 환경변수 GH_TOKEN → Colab 보안 비밀 → 입력창 순서로 인증합니다.
# Fine-grained token: SongYunu/moveBoxes → Contents: Read and write.
# 이 저장소는 공개이므로 여기에 올린 모델·로그·영상도 공개됩니다.
import os
# 개인 사본에서 직접 지정할 경우 아래 한 줄의 주석을 풀어 사용하세요.
# os.environ['GH_TOKEN'] = '본인 토큰'
experiment.connect()
experiment.show_results()


GitHub 토큰 입력 (이 런타임에서만 사용): ··········
GitHub 복원: 45 files
로컬 작업 경로: /content/moveboxes_runs/moveboxes_stage_chunk_deadline_v1_benchmark
실행 노트북 사본: /content/moveboxes_runs/moveboxes_stage_chunk_deadline_v1_benchmark/moveboxes_stages_colab.ipynb
새 런타임에서는 01~03 셀로 결과를 복원하고, 학습·평가는 04~05 셀 준비 후 실행합니다.
GitHub 백업 완료: common (45 files)
단계 ACT: 학습한 완료/복구 판단 + 집기/운반/놓기 행동, 별도 실험
easy   |      -- | 모델 없음 | 학습완료 기록 False
medium |      -- | 모델 없음 | 학습완료 기록 False
hard   |      -- | 모델 없음 | 학습완료 기록 False
가중 점수: 0.0000 / 미평가: ['easy', 'medium', 'hard']


In [4]:
# 04 · 새 런타임마다 환경 설치
experiment.install()


실행: nvidia-smi
전체 로그: /content/moveboxes_runs/moveboxes_stage_chunk_deadline_v1_benchmark/sessions/20260915_024846_d99bca/gpu.log
완료
실행: git clone https://github.com/marso-robotics/berlin-marso-hackathon.git /content/berlin-marso-hackathon
전체 로그: /content/moveboxes_runs/moveboxes_stage_chunk_deadline_v1_benchmark/sessions/20260915_024846_d99bca/commands.log
완료
실행: /usr/bin/python3 -m pip install mani-skill==3.0.1 sapien==3.0.3 diffusers==0.38.0 hydra-core omegaconf gymnasium tyro h5py kagglehub tensorboard matplotlib transforms3d imageio[ffmpeg]
전체 로그: /content/moveboxes_runs/moveboxes_stage_chunk_deadline_v1_benchmark/sessions/20260915_024846_d99bca/install.log
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 MB 17.3 MB/s eta 0:00:00
Successfully uninstalled diffusers-0.39.0
완료
실행: /usr/bin/python3 -m pip install -e /content/berlin-marso-hackathon
전체 로그: /content/moveboxes_runs/moveboxes_stage_chunk_deadline_v1_benchmark/sessions/20260915_024846_d99bca/install_repo.log
완료
설치 완료. 다음 

In [5]:
# 05 · GitHub 데이터 다운로드·검증 / GPU와 정책 실행 확인
experiment.prepare_data()
experiment.check_runtime()


GitHub 데이터 다운로드: state (65.4 MiB)
easy 200 demos
medium 200 demos
hard 200 demos
GitHub 백업 완료: common (51 files)
실행: /usr/bin/python3 /content/berlin-marso-hackathon/test_policy_runtime.py
전체 로그: /content/moveboxes_runs/moveboxes_stage_chunk_deadline_v1_benchmark/sessions/20260915_024846_d99bca/policy_check.log
Stage ACT 54: backward/load/action/reset OK (cuda)
완료
실행: /usr/bin/python3 -c import torch
from warehouse_sort.utils import compose_cfg, make_env
assert torch.cuda.is_available(), 'GPU runtime required'
for level, dims in [('easy',54),('medium',72),('hard',90)]:
    cfg = compose_cfg(['difficulty='+level, 'num_envs=1'])
    env, _ = make_env(cfg, 'state', cfg.randomization, num_envs=1, render_mode='rgb_array')
    try:
        obs, _ = env.reset(seed=42)
        assert tuple(obs.shape)==(1,dims)
        env.step(torch.zeros((1,4),device='cuda'))
        assert env.render() is not None
        print(level, 'GPU/render OK')
    finally:
        env.close()

전체 로그: /content/movebox

## 실행 순서

03 셀은 `GH_TOKEN` 환경변수, Colab Secrets, 비공개 입력창 순서로 GitHub 토큰을 받습니다. 같은 `run_name`의 **현재 학습 상태만** GitHub Release에서 복원합니다. 과거 best checkpoint를 자동으로 가져오지 않습니다.

04~05에서 T4 환경과 state dataset을 준비합니다. 각 난이도는 recovery 수집 후 Stage ACT를 학습합니다. 수집은 매 시도, 학습은 매 1,000 iteration마다 GitHub에 동기화됩니다. 런타임이 끊기면 새 런타임에서 01~05와 해당 난이도의 수집·학습 셀을 다시 실행하면 이어집니다.


In [6]:
# 07 · EASY recovery 수집 · 매 시도 GitHub 저장, 재실행 시 이어서 진행
experiment.collect('easy')


실행: /usr/bin/python3 stage_collect.py /content/moveboxes_runs/moveboxes_stage_chunk_deadline_v1_benchmark/easy/collection_job.json
전체 로그: /content/moveboxes_runs/moveboxes_stage_chunk_deadline_v1_benchmark/easy/collection.log
GitHub 백업 완료: easy (4 files)
GitHub 백업 완료: easy (6 files)
GitHub 백업 완료: easy (8 files)
GitHub 백업 완료: easy (10 files)
GitHub 백업 완료: easy (12 files)
GitHub 백업 완료: easy (14 files)
GitHub 백업 완료: easy (16 files)
GitHub 백업 완료: easy (18 files)
완료
GitHub 백업 완료: easy (19 files)
GitHub 백업 완료: common (53 files)


In [7]:
# 08 · EASY State Stage ACT 학습 · 로그 표시 + 매 checkpoint 완전 복구 저장
# latest.pt에는 model, optimizer, AMP scaler, sampling/CPU/CUDA RNG가 함께 저장됩니다.
experiment.train('easy')


실행: /usr/bin/python3 /content/berlin-marso-hackathon/stage_train.py /content/moveboxes_runs/moveboxes_stage_chunk_deadline_v1_benchmark/easy/stage_train_job.json
전체 로그: /content/moveboxes_runs/moveboxes_stage_chunk_deadline_v1_benchmark/easy/train.log
Stage ACT: 1.28M parameters / cuda / AMP=True / resume 0/12000
228/12000 loss=0.4994 22.7 it/s ETA=8.6 min
505/12000 loss=0.2608 25.2 it/s ETA=7.6 min
770/12000 loss=0.2077 25.6 it/s ETA=7.3 min
GitHub 백업 완료: easy (28 files)
1254/12000 loss=0.1386 20.8 it/s ETA=8.6 min
1526/12000 loss=0.0958 21.7 it/s ETA=8.0 min
1787/12000 loss=0.0659 22.3 it/s ETA=7.6 min
GitHub 백업 완료: easy (28 files)
2262/12000 loss=0.0502 21.6 it/s ETA=7.5 min
2510/12000 loss=0.0537 21.9 it/s ETA=7.2 min
2751/12000 loss=0.0496 22.0 it/s ETA=7.0 min
GitHub 백업 완료: easy (28 files)
3244/12000 loss=0.0462 21.7 it/s ETA=6.7 min
3500/12000 loss=0.0424 21.9 it/s ETA=6.5 min
3776/12000 loss=0.0260 22.2 it/s ETA=6.2 min
GitHub 백업 완료: easy (28 files)
4285/12000 loss=0.0273 22.0 

In [8]:
# 09 · MEDIUM recovery 수집 · 매 시도 GitHub 저장, 재실행 시 이어서 진행
experiment.collect('medium')


실행: /usr/bin/python3 stage_collect.py /content/moveboxes_runs/moveboxes_stage_chunk_deadline_v1_benchmark/medium/collection_job.json
전체 로그: /content/moveboxes_runs/moveboxes_stage_chunk_deadline_v1_benchmark/medium/collection.log
GitHub 백업 완료: medium (4 files)
GitHub 백업 완료: medium (5 files)
GitHub 백업 완료: medium (5 files)
GitHub 백업 완료: medium (6 files)
GitHub 백업 완료: medium (7 files)
GitHub 백업 완료: medium (8 files)
GitHub 백업 완료: medium (9 files)
GitHub 백업 완료: medium (10 files)
GitHub 백업 완료: medium (11 files)
GitHub 백업 완료: medium (12 files)
GitHub 백업 완료: medium (13 files)
GitHub 백업 완료: medium (14 files)
GitHub 백업 완료: medium (15 files)
GitHub 백업 완료: medium (16 files)
GitHub 백업 완료: medium (17 files)
GitHub 백업 완료: medium (18 files)
GitHub 백업 완료: medium (19 files)
완료
GitHub 백업 완료: medium (19 files)


In [9]:
# 10 · MEDIUM State Stage ACT 학습 · 로그 표시 + 매 checkpoint 완전 복구 저장
# latest.pt에는 model, optimizer, AMP scaler, sampling/CPU/CUDA RNG가 함께 저장됩니다.
experiment.train('medium')


실행: /usr/bin/python3 /content/berlin-marso-hackathon/stage_train.py /content/moveboxes_runs/moveboxes_stage_chunk_deadline_v1_benchmark/medium/stage_train_job.json
전체 로그: /content/moveboxes_runs/moveboxes_stage_chunk_deadline_v1_benchmark/medium/train.log
228/20000 loss=0.5597 22.7 it/s ETA=14.5 min
464/20000 loss=0.2890 23.2 it/s ETA=14.1 min
714/20000 loss=0.2300 23.7 it/s ETA=13.5 min
963/20000 loss=0.2025 24.0 it/s ETA=13.2 min
GitHub 백업 완료: medium (28 files)
1245/20000 loss=0.1752 19.6 it/s ETA=16.0 min
1490/20000 loss=0.1472 20.3 it/s ETA=15.2 min
1741/20000 loss=0.1774 20.8 it/s ETA=14.6 min
GitHub 백업 완료: medium (28 files)
2252/20000 loss=0.0613 20.6 it/s ETA=14.4 min
2503/20000 loss=0.0906 20.9 it/s ETA=13.9 min
2789/20000 loss=0.0581 21.5 it/s ETA=13.3 min
GitHub 백업 완료: medium (28 files)
3280/20000 loss=0.0530 21.2 it/s ETA=13.1 min
3538/20000 loss=0.0372 21.5 it/s ETA=12.8 min
3791/20000 loss=0.0305 21.7 it/s ETA=12.4 min
GitHub 백업 완료: medium (28 files)
4242/20000 loss=0.0246

In [10]:
# 11 · HARD recovery 수집 · 매 시도 GitHub 저장, 재실행 시 이어서 진행
experiment.collect('hard')


실행: /usr/bin/python3 stage_collect.py /content/moveboxes_runs/moveboxes_stage_chunk_deadline_v1_benchmark/hard/collection_job.json
전체 로그: /content/moveboxes_runs/moveboxes_stage_chunk_deadline_v1_benchmark/hard/collection.log
GitHub 백업 완료: hard (3 files)
GitHub 백업 완료: hard (4 files)
GitHub 백업 완료: hard (4 files)
GitHub 백업 완료: hard (5 files)
GitHub 백업 완료: hard (6 files)
GitHub 백업 완료: hard (7 files)
GitHub 백업 완료: hard (8 files)
GitHub 백업 완료: hard (8 files)
GitHub 백업 완료: hard (9 files)
GitHub 백업 완료: hard (10 files)
GitHub 백업 완료: hard (11 files)
GitHub 백업 완료: hard (12 files)
GitHub 백업 완료: hard (13 files)
GitHub 백업 완료: hard (14 files)
GitHub 백업 완료: hard (15 files)
GitHub 백업 완료: hard (16 files)
GitHub 백업 완료: hard (17 files)
GitHub 백업 완료: hard (18 files)
GitHub 백업 완료: hard (19 files)
완료
GitHub 백업 완료: hard (19 files)


In [11]:
# 12 · HARD State Stage ACT 학습 · 로그 표시 + 매 checkpoint 완전 복구 저장
# latest.pt에는 model, optimizer, AMP scaler, sampling/CPU/CUDA RNG가 함께 저장됩니다.
experiment.train('hard')


실행: /usr/bin/python3 /content/berlin-marso-hackathon/stage_train.py /content/moveboxes_runs/moveboxes_stage_chunk_deadline_v1_benchmark/hard/stage_train_job.json
전체 로그: /content/moveboxes_runs/moveboxes_stage_chunk_deadline_v1_benchmark/hard/train.log
235/30000 loss=0.5839 23.4 it/s ETA=21.2 min
483/30000 loss=0.3409 24.1 it/s ETA=20.4 min
760/30000 loss=0.2784 25.2 it/s ETA=19.3 min
GitHub 백업 완료: hard (28 files)
1246/30000 loss=0.2306 19.9 it/s ETA=24.0 min
1495/30000 loss=0.1704 20.6 it/s ETA=23.0 min
1743/30000 loss=0.1687 21.1 it/s ETA=22.3 min
GitHub 백업 완료: hard (28 files)
2240/30000 loss=0.1568 20.8 it/s ETA=22.2 min
2488/30000 loss=0.1758 21.1 it/s ETA=21.7 min
2729/30000 loss=0.0763 21.4 it/s ETA=21.3 min
2948/30000 loss=0.1496 21.4 it/s ETA=21.1 min
3058/30000 loss=0.1280 20.7 it/s ETA=21.7 min
3293/30000 loss=0.0926 20.9 it/s ETA=21.3 min
3543/30000 loss=0.0537 21.1 it/s ETA=20.9 min
3790/30000 loss=0.1282 21.3 it/s ETA=20.5 min
GitHub 백업 완료: hard (28 files)
4237/30000 loss=0

In [15]:
# 현재 run의 학습 checkpoint에 단일 stage-aware 실행 정책 적용
import hashlib, importlib.metadata, os, shutil, subprocess, sys
from pathlib import Path
import torch

import json
RUN_DIR = Path(experiment.run_dir)
CANDIDATE = RUN_DIR/'integrated_candidate'
CHECKPOINT_OVERRIDES = {'easy':'', 'medium':'', 'hard':''}
STAGE_HORIZONS = dict(pick=2, carry=6, place=2, done=1)
GRIPPER_MARGIN = .5
GRIPPER_CONFIRM_STEPS = 2
MAX_STEPS = 200
SMOKE_SEED = 61000
DIMS = {'easy':54, 'medium':72, 'hard':90}

# 경로를 따로 지정하지 않으면 이 run에서 학습된 best_val/latest만 사용합니다.
selected = {}
for level in DIMS:
    override = CHECKPOINT_OVERRIDES[level]
    if override:
        checkpoint = Path(override).expanduser().resolve()
    else:
        folder = RUN_DIR/level/'checkpoints'
        checkpoint = next((p for p in (folder/'best_val.pt', folder/'latest.pt') if p.is_file()), None)
    if checkpoint is not None:
        selected[level] = checkpoint
if not selected:
    raise FileNotFoundError('먼저 하나 이상의 experiment.train(level) 셀을 실행하세요.')

def sha256(path):
    with Path(path).open('rb') as handle:
        return hashlib.file_digest(handle, 'sha256').hexdigest()

CANDIDATE.mkdir(parents=True, exist_ok=True)
for source in ('stage_chunk_policy.py','stage_policy.py','stage_model.py','stage_schema.py'):
    shutil.copy2(PROJECT/'ver2/stages'/source, CANDIDATE/source)
shutil.copy2(PROJECT/'ver2/act_v2_model.py', CANDIDATE/'act_v2_model.py')

manifest = dict(policy='state_stage_act_chunk_fsm', project_commit=CFG['project_commit'],
                official_commit=CFG['repo_commit'], run_name=CFG['run_name'], levels={})
level_lines = []
for level, checkpoint in selected.items():
    saved = torch.load(checkpoint, map_location='cpu', weights_only=True)
    if saved.get('format') != 'moveboxes-stage-act-v1':
        raise ValueError(f'{level}: Stage ACT checkpoint가 아닙니다: {saved.get("format")}')
    model_config = saved['model_config']
    if model_config.get('state_dim') != DIMS[level] or model_config.get('chunk_size', 0) < 6:
        raise ValueError(f'{level}: state dimension 또는 trained chunk size 불일치')
    sidecar = checkpoint.parent/'policy_config.json'
    if not sidecar.is_file():
        raise FileNotFoundError(sidecar)
    policy = json.loads(sidecar.read_text(encoding='utf-8'))
    if policy.get('model_config') != model_config:
        raise ValueError(f'{level}: checkpoint/sidecar architecture 불일치')
    policy.update(model_config=model_config, stage_aware_chunk=True,
                  stage_horizons=STAGE_HORIZONS, gripper_fsm=True,
                  gripper_margin=GRIPPER_MARGIN,
                  gripper_confirm_steps=GRIPPER_CONFIRM_STEPS,
                  auto_reset_steps=MAX_STEPS-1)
    policy.pop('act_horizon', None)
    policy.pop('num_inference_steps', None)
    target_dir = CANDIDATE/'checkpoints'/level
    target_dir.mkdir(parents=True, exist_ok=True)
    target = target_dir/'model.pt'
    shutil.copy2(checkpoint, target)
    (target_dir/'policy_config.json').write_text(json.dumps(policy, indent=2), encoding='utf-8')
    if sha256(target) != sha256(checkpoint):
        raise RuntimeError(f'{level}: checkpoint 사본 해시 불일치')
    manifest['levels'][level] = dict(source=str(checkpoint), checkpoint_sha256=sha256(target),
                                     step=saved.get('step'), model_config=model_config,
                                     policy_config=policy)
    level_lines.append(f'    {level}: {{ checkpoint: checkpoints/{level}/model.pt }}')

submission = 'team: '+json.dumps(CFG['team'])+'\nstate:\n  policy: stage_chunk_policy:load_policy\n  levels:\n'+'\n'.join(level_lines)+'\n'
(CANDIDATE/'submission.yaml').write_text(submission, encoding='utf-8')
(CANDIDATE/'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('현재 run checkpoint:', {k:str(v) for k,v in selected.items()})
print('단일 integrated candidate:', CANDIDATE)


현재 run checkpoint: {'easy': '/content/moveboxes_runs/moveboxes_stage_chunk_deadline_v1_benchmark/easy/checkpoints/best_val.pt', 'medium': '/content/moveboxes_runs/moveboxes_stage_chunk_deadline_v1_benchmark/medium/checkpoints/best_val.pt', 'hard': '/content/moveboxes_runs/moveboxes_stage_chunk_deadline_v1_benchmark/hard/checkpoints/best_val.pt'}
단일 integrated candidate: /content/moveboxes_runs/moveboxes_stage_chunk_deadline_v1_benchmark/integrated_candidate


In [16]:
# 실제 simulator state/action/buffer/reset sanity check
from stage_chunk_policy import load_policy
from warehouse_sort.utils import compose_cfg, make_env

for level in selected:
    cfg = compose_cfg(['difficulty='+level, 'obs_mode=state', 'max_episode_steps='+str(MAX_STEPS)],
                      config_dir=str(Path(CFG['repo_dir'])/'conf'))
    env, _ = make_env(cfg, 'state', cfg.randomization, num_envs=1)
    try:
        obs, _ = env.reset(seed=SMOKE_SEED)
        assert tuple(obs.shape) == (1, DIMS[level])
        agent = load_policy(CANDIDATE/'checkpoints'/level/'model.pt', obs,
                            env.single_action_space, 'cuda')
        with torch.no_grad():
            first = agent.act(obs)
        assert first.shape == (1,4) and torch.isfinite(first).all()
        assert first.abs().max() <= 1 and first[0,3].item() in (-1.,1.)
        assert agent.action_buffer.shape[1] == manifest['levels'][level]['model_config']['chunk_size']
        agent.reset()
        assert not agent.history and agent.action_buffer is None and agent.grip_state is None
        obs2, _ = env.reset(seed=SMOKE_SEED)
        with torch.no_grad():
            repeated = agent.act(obs2)
        torch.testing.assert_close(first, repeated, rtol=0, atol=0)
        agent.step = MAX_STEPS-1
        agent.history.append(torch.full_like(obs2, 123.))
        agent.action_buffer.fill_(123.)
        with torch.no_grad():
            boundary = agent.act(obs2)
        torch.testing.assert_close(first, boundary, rtol=0, atol=0)
        assert agent.step == 1
        env.step(repeated)
        print(level, 'state/action/buffer/manual+official reset/env.step OK')
    finally:
        env.close()


easy state/action/buffer/manual+official reset/env.step OK
medium state/action/buffer/manual+official reset/env.step OK
hard state/action/buffer/manual+official reset/env.step OK


In [17]:
# 공식 eval.py · 1 episode smoke
RESULTS = RUN_DIR
SMOKE_CONFIG = RUN_DIR/'integrated_smoke_eval.yaml'
SMOKE_CONFIG.write_text('eval:\n  n_episodes: 1\n  seeds: ['+str(SMOKE_SEED)+']\n', encoding='utf-8')
UPSTREAM = Path(CFG['repo_dir'])

def run_official(level, eval_config, label):
    output = RUN_DIR/level/'integrated_official_eval'/label
    output.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, str(UPSTREAM/'eval.py'), 'difficulty='+level,
        'obs_mode=state', 'policy=stage_chunk_policy:load_policy',
        'checkpoint='+str(CANDIDATE/'checkpoints'/level/'model.pt'),
        'eval_config='+str(eval_config), 'max_episode_steps='+str(MAX_STEPS),
        'hydra.run.dir='+str(output)]
    child_env = dict(os.environ)
    child_env['PYTHONPATH'] = str(CANDIDATE)+os.pathsep+str(UPSTREAM)+os.pathsep+child_env.get('PYTHONPATH','')
    log = output/'official_eval.log'
    with log.open('w', encoding='utf-8') as handle:
        process = subprocess.Popen(command, cwd=UPSTREAM, env=child_env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, errors='replace', bufsize=1)
        for line in process.stdout:
            print(line, end='')
            handle.write(line)
            handle.flush()
        code = process.wait()
    if code:
        raise RuntimeError(f'official eval failed ({level}); {log} 확인')
    experiment.sync_level(level)
    return log

for level in selected:
    run_official(level, SMOKE_CONFIG, 'smoke')


/usr/local/lib/python3.12/dist-packages/sapien/_vulkan_tricks.py:42: UserWarning: Failed to find system libvulkan. Fallback to SAPIEN builtin libvulkan.
  warn("Failed to find system libvulkan. Fallback to SAPIEN builtin libvulkan.")
[eval] git=6048f33217f26ae39009a812f53c81171517f393
----------------------------------------------------------------------
difficulty:
  name: easy
  num_parcels: 2
  fixed_poses: true
randomization:
  parcel_pose:
    xy_jitter:
    - 0.0
    - 0.0
    yaw_jitter:
    - 0.0
    - 0.0
  bin_position:
    side_swap_prob: 0.0
    xy_jitter:
    - 0.0
    - 0.0
seed: 0
num_envs: 8
device: cuda
control_mode: pd_ee_delta_pos
max_episode_steps: 200
obs_mode: state
camera:
  width: 128
  height: 128
obs_camera: scene
checkpoint: /content/moveboxes_runs/moveboxes_stage_chunk_deadline_v1_benchmark/integrated_candidate/checkpoints/easy/model.pt
eval_config: /content/moveboxes_runs/moveboxes_stage_chunk_deadline_v1_benchmark/integrated_smoke_eval.yaml
policy: stage_c

AssetUploadError: GitHub asset upload failed (422); local files are retained.

In [18]:
# 공식 conf/eval/default.yaml 평가 · 자체 metric/승자 선택 없음
OFFICIAL_EVAL_CONFIG = UPSTREAM/'conf/eval/default.yaml'
if not OFFICIAL_EVAL_CONFIG.is_file():
    raise FileNotFoundError(OFFICIAL_EVAL_CONFIG)
for level in selected:
    run_official(level, OFFICIAL_EVAL_CONFIG, 'default')
print('공식 raw logs/videos:', {level:str(RUN_DIR/level/'integrated_official_eval'/'default') for level in selected})


/usr/local/lib/python3.12/dist-packages/sapien/_vulkan_tricks.py:42: UserWarning: Failed to find system libvulkan. Fallback to SAPIEN builtin libvulkan.
  warn("Failed to find system libvulkan. Fallback to SAPIEN builtin libvulkan.")
[eval] git=6048f33217f26ae39009a812f53c81171517f393
----------------------------------------------------------------------
difficulty:
  name: easy
  num_parcels: 2
  fixed_poses: true
randomization:
  parcel_pose:
    xy_jitter:
    - 0.0
    - 0.0
    yaw_jitter:
    - 0.0
    - 0.0
  bin_position:
    side_swap_prob: 0.0
    xy_jitter:
    - 0.0
    - 0.0
seed: 0
num_envs: 8
device: cuda
control_mode: pd_ee_delta_pos
max_episode_steps: 200
obs_mode: state
camera:
  width: 128
  height: 128
obs_camera: scene
checkpoint: /content/moveboxes_runs/moveboxes_stage_chunk_deadline_v1_benchmark/integrated_candidate/checkpoints/easy/model.pt
eval_config: /content/berlin-marso-hackathon/conf/eval/default.yaml
policy: stage_chunk_policy:load_policy
----------------

KeyboardInterrupt: 

In [ ]:
# 단일 candidate ZIP 생성 및 브라우저 다운로드
check = """import json,sys,torch
from pathlib import Path
from types import SimpleNamespace
from stage_chunk_policy import load_policy
root=Path(sys.argv[1])
manifest=json.loads((root/'manifest.json').read_text())
for level,row in manifest['levels'].items():
 p=root/'checkpoints'/level/'model.pt'
 agent=load_policy(p,torch.zeros(1,row['model_config']['state_dim']),SimpleNamespace(shape=(4,)),'cpu')
 a=agent.act(torch.zeros(1,row['model_config']['state_dim']))
 assert a.shape==(1,4) and torch.isfinite(a).all() and a.abs().max()<=1
 agent.reset()
 print(level,'candidate import/action/reset OK')
"""
subprocess.run([sys.executable, '-c', check, str(CANDIDATE)], cwd=CANDIDATE, check=True)
archive = shutil.make_archive(str(CANDIDATE), 'zip', CANDIDATE)
from google.colab import files
print('candidate ZIP:', archive)
files.download(archive)


In [20]:
from pathlib import Path
from IPython.display import Video, display

root = Path(experiment.run_dir)

for level in ('easy', 'medium', 'hard'):
    folder = root / level / 'integrated_official_eval/smoke/videos'
    videos = sorted(
        folder.rglob('*.mp4'),
        key=lambda p: p.stat().st_mtime
    )

    if not videos:
        print(f'{level}: 아직 생성된 영상이 없습니다.')
        continue

    print(f'▶ {level.upper()}')
    display(Video(filename=str(videos[-1]), embed=True, width=960))

▶ EASY


TypeError: stat: path should be string, bytes, os.PathLike or integer, not NoneType

In [21]:
import shutil
from pathlib import Path
from google.colab import files

run_dir = Path(experiment.run_dir)
checkpoints = list(run_dir.glob('*/checkpoints/latest.pt'))
assert checkpoints, f'checkpoint가 없습니다: {run_dir}'

archive = shutil.make_archive(
    '/content/' + run_dir.name + '_backup',
    'zip',
    root_dir=run_dir.parent,
    base_dir=run_dir.name,
)

print('백업 checkpoint:', [str(p) for p in checkpoints])
files.download(archive)

백업 checkpoint: ['/content/moveboxes_runs/moveboxes_stage_chunk_deadline_v1_benchmark/hard/checkpoints/latest.pt', '/content/moveboxes_runs/moveboxes_stage_chunk_deadline_v1_benchmark/easy/checkpoints/latest.pt', '/content/moveboxes_runs/moveboxes_stage_chunk_deadline_v1_benchmark/medium/checkpoints/latest.pt']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>